While runnable as a notebook, this is more convenient in some ways to run like a script - e.g.:

``pixi run jupyter nbconvert --to python psf_training_metaparameter_scan.ipynb --stdout | systemd-inhibit --what=idle:sleep --why="long-running data analysis" pixi run -e gpu python -``

In [ ]:
from pathlib import Path

import nirspectroperf
import papermill

In [ ]:
output_dir = Path('psf_training_metaparameter_scan')
output_dir.mkdir(exist_ok=True)

psf_model_template_path = nirspectroperf.get_template_notebooks()['train_psf_models']

inspected = papermill.inspect_notebook(psf_model_template_path)
inspected

# Single parameter variation

In [ ]:
template_dict = {'save_model_fn_pattern': True, 'pbar':0.05} # if save_model_fn_pattern is True, replace below
template_dict['template_data_file_fn'] = '../' + eval(inspected['template_data_file_fn']['default'])
parameter_dicts = [template_dict] # first one is fiducial

For the below, we only vary one parameter at a time, and leave the others at their defaults. So these *exclude* the fiduicial case (i.e., all defaults)

In [ ]:
for nepochs in [20000, 60000]:
    parameter_dicts.append(template_dict | {'nepochs': nepochs})

In [ ]:
for batch_size in [1024, 4096]:
    parameter_dicts.append(template_dict | {'batch_size': batch_size})

In [ ]:
for loss_meantologweight in [0.1, 0.9, 'progressive']:
    parameter_dicts.append(template_dict | {'loss_meantologweight': loss_meantologweight})

In [ ]:
for minlr in [1e-5, 1e-4, 1e-3]:
    for peaklr in [1e-3, 5e-3]:
        parameter_dicts.append(template_dict | {'minlr': minlr, 'peaklr': peaklr})

In [ ]:
for s0 in [5, 10, 15]:
    for w0 in [5, 10, 15]:
        if s0 == eval(inspected['s0']['default']) and w0 == eval(inspected['w0']['default']):
            continue
        parameter_dicts.append(template_dict | {'s0': s0, 'w0': w0})

In [ ]:
def nb_name_from_pdict(pdict):
    dsub = {k: v for k, v in pdict.items() if k not in template_dict.keys()}
    subname = 'fiducial'

    if dsub != {}:
        tojoin = []
        for k, v in dsub.items():
            if isinstance(v, float):
                tojoin.append(f'{k}{v:.1e}')
            else:
                tojoin.append(f'{k}{v}')

        rawstr = '_'.join(tojoin)
        subname = rawstr.replace('.','p').replace('-','m')
        
    return 'psf_training_' + subname + '.ipynb'

assert len(set([nb_name_from_pdict(pdict) for pdict in parameter_dicts])) == len(parameter_dicts), "Duplicate names found"

len(parameter_dicts)

In [ ]:
print('Will make the following notebooks:', [nb_name_from_pdict(pdict) for pdict in parameter_dicts])

Note that if the PSF cache is not already populated the first notebook will take a lot longer to run.

In [ ]:
donenbs = []

for pdict in parameter_dicts:
    nm = output_dir / nb_name_from_pdict(pdict)
    if nm.exists():
        print('Notebook', nm, 'already exists, skipping.')
        donenbs.append(None)
    else:
        print('Notebook', nm, 'does not exist, executing.')
        pdict = pdict.copy()

        pdict['plot_save_fn_pattern'] = nm.with_stem('{subname}-' + nm.stem).with_suffix('.png').name
        if pdict.get('save_model_fn_pattern', False) is True:
            pdict['save_model_fn_pattern'] = nm.with_suffix('.model').name

        nb = papermill.execute_notebook(psf_model_template_path, nm, 
                                        parameters=pdict, 
                                        cwd=output_dir,
                                        request_save_on_cell_execute=True,
                                        autosave_cell_every=30,
                                        progress_bar=True)
        donenbs.append(nb)
    
donenbs

The below is useful for comparison in that it produces videos like in the template notebook so are useful for comparison.

In [ ]:
import numpy as np
from nirspectroperf.psf import NIRSpecPSF
import astropy.units as u

from matplotlib import pyplot as plt
from matplotlib import animation

In [ ]:
imshowkwargs = dict(origin='lower', interpolation='nearest')
intervalms = 200
vmax = 0.035
nbins = 6

# pick a random subset of PSFs to animate
fraction = 0.03

cache_psfs = np.load(output_dir / eval(inspected['cache_fn']['default']), allow_pickle=True)

fig, ax = plt.subplots(figsize=(5, 5))

pxscl = cache_psfs['example_header'].item()['PIXELSCL']
pxwid = NIRSpecPSF.msa_slit_width.to_value(u.arcsec)/pxscl
pxhei = NIRSpecPSF.msa_slit_height.to_value(u.arcsec)/pxscl

ax.add_artist(plt.Rectangle((100-pxwid/2, 100-pxhei/2), pxwid, pxhei, fill=False, alpha=0.5, color='r'))

def fanim(ilog):
    idx, dolog = ilog
    x = cache_psfs['xoffs'][idx]
    y = cache_psfs['yoffs'][idx]
    thispsf = cache_psfs['psflux'][idx]

    psfsum = thispsf.sum()

    # normalize the same as the model
    thispsf = thispsf/psfsum
    
    logstr = ''
    thisvmax = vmax
    if dolog:
        thispsf = np.log(thispsf)
        logstr = 'logflux, '
        thisvmax = np.log(vmax)
    ax.imshow(thispsf, **imshowkwargs, vmax=thisvmax)
    ax.set_title(f'{logstr}X={x:.3f}, Y={y:.3f}, sum={psfsum:.3}', color=plt.colormaps.get_cmap('viridis')(psfsum))

    fig.tight_layout()

def idx_sorted_by_x_in_ybins(xs, ys, nbins, reverse_every_other=False):
    idxs = np.arange(len(xs))
    binedges = np.linspace(np.min(ys), np.max(ys), nbins)
    binnum = np.digitize(ys, binedges)

    binlist = []
    if reverse_every_other < 0:
        reverse_mod = 0
    elif reverse_every_other > 0:
        reverse_mod = 1
    else:
        reverse_mod = None
    for i, num in enumerate(np.unique(binnum)):
        sorti = np.argsort(xs[idxs[binnum==num]])
        idxs_sorted = idxs[binnum==num][sorti]
        if reverse_mod is not None and i % 2 == reverse_mod:
            idxs_sorted = idxs_sorted[::-1]
        binlist.append(idxs_sorted)

    return binlist


idxs = []
for idxlist in idx_sorted_by_x_in_ybins(cache_psfs['xoffs'], cache_psfs['yoffs'], nbins, reverse_every_other=1)[::-1]:
    idxs.extend(idxlist)
idxs = np.array(idxs)

if fraction < 1:
    idxs = idxs[np.random.rand(len(idxs)) < fraction]

anim = animation.FuncAnimation(fig, fanim, frames=list(zip(idxs, [False]*len(idxs))), interval=intervalms)
anim.save(output_dir / 'true_psf.mp4')

animlog = animation.FuncAnimation(fig, fanim, frames=list(zip(idxs, [True]*len(idxs))), interval=intervalms)
animlog.save(output_dir / 'true_logpsf.mp4')

plt.close(fig)